In [9]:
from __future__ import annotations

import ast
import math
import re
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# CONFIG
# ============================================================

CSV_PATH = Path("Results/logs/adam/qaoa_results_adam_20260617_173540_pid708337.csv")

ADJ_PATH_CANDIDATES = [
    Path("Code/HOG_graphs/triangleFree_connected_4to8vertices_176instances_AdjacencyList.txt"),
    Path("HOG_graphs/triangleFree_connected_4to8vertices_176instances_AdjacencyList.txt"),
]

OUT_PATH = Path("Results/logs/adam/L2M2_runtime_estimate_by_instance_seed_depth.csv")
SUMMARY_PATH = Path("Results/logs/adam/L2M2_runtime_estimate_by_graph_summary.csv")

NUM_REPEATS = 10
DEPTHS = [1, 3, 5]

BASE_SEED = 42
SEED_STEP = 1_000_000

LASSERRE_LEVEL = 2
INITIAL_SOLVER_LEVEL_M = 2

# HOG indices are 0-based in the launcher.
FIRST_HOG_INDEX = 51

# Used only for printed wall-time summary.
WORKERS = 5

# Per-instance timeout in your config. Values above this are almost certainly not per-instance QAOA runtimes.
MAX_VALID_RUNTIME_SECONDS = 10_800

# Leave as None unless auto-detection chooses the wrong column.
VERTEX_COL_OVERRIDE = None
HOG_INDEX_COL_OVERRIDE = None
REPEAT_COL_OVERRIDE = None
DEPTH_COL_OVERRIDE = None
TIME_COL_OVERRIDE = None


# ============================================================
# PATH CHECKS
# ============================================================

ADJ_PATH = next((p for p in ADJ_PATH_CANDIDATES if p.exists()), None)

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH.resolve()}")

if ADJ_PATH is None:
    raise FileNotFoundError(
        "Adjacency-list file not found. Tried:\n"
        + "\n".join(str(p.resolve()) for p in ADJ_PATH_CANDIDATES)
    )

print("Using CSV:")
print(CSV_PATH.resolve())
print()
print("Using adjacency-list file:")
print(ADJ_PATH.resolve())
print()


# ============================================================
# GRAPH PARSING
# ============================================================

def read_hog_blocks(path: Path) -> list[str]:
    """
    Match the launcher semantics: one graph block is separated by blank lines.
    HOG indices are therefore 0-based block indices.
    """
    blocks = []
    current = []

    for line in path.read_text().splitlines():
        if line.strip() == "":
            if current:
                blocks.append("\n".join(current))
                current = []
        else:
            current.append(line)

    if current:
        blocks.append("\n".join(current))

    return blocks


def parse_graph_block(block: str) -> tuple[int, int | float]:
    """
    Return (vertices, edges).

    Supports:
      - adjacency dict: {0: [1, 2], 1: [0]}
      - adjacency list: [[1, 2], [0], ...]
      - edge list: [(0, 1), (1, 2)]
    """
    block = block.strip()
    if not block:
        return 0, 0

    try:
        obj = ast.literal_eval(block)

        if isinstance(obj, dict):
            nodes = set()
            edges = set()

            for u_raw, nbrs in obj.items():
                u = int(u_raw)
                nodes.add(u)

                if isinstance(nbrs, (list, tuple, set)):
                    for v_raw in nbrs:
                        v = int(v_raw)
                        nodes.add(v)
                        a, b = sorted((u, v))
                        if a != b:
                            edges.add((a, b))

            vertices = max(nodes) + 1 if nodes else len(obj)
            return vertices, len(edges)

        if isinstance(obj, (list, tuple)):
            if all(isinstance(x, (list, tuple, set)) for x in obj):
                nodes = set(range(len(obj)))
                edges = set()

                for u, nbrs in enumerate(obj):
                    for v_raw in nbrs:
                        v = int(v_raw)
                        nodes.add(v)
                        a, b = sorted((u, v))
                        if a != b:
                            edges.add((a, b))

                vertices = max(nodes) + 1 if nodes else len(obj)
                return vertices, len(edges)

            if all(isinstance(x, (list, tuple)) and len(x) >= 2 for x in obj):
                nodes = set()
                edges = set()

                for edge in obj:
                    u, v = int(edge[0]), int(edge[1])
                    nodes.update([u, v])
                    a, b = sorted((u, v))
                    if a != b:
                        edges.add((a, b))

                vertices = max(nodes) + 1 if nodes else 0
                return vertices, len(edges)

    except Exception:
        pass

    nums = [int(x) for x in re.findall(r"\d+", block)]
    vertices = max(nums) + 1 if nums else 0
    return vertices, math.nan


blocks = read_hog_blocks(ADJ_PATH)

graph_rows = []
for hog_index, block in enumerate(blocks):
    vertices, edges = parse_graph_block(block)

    graph_rows.append(
        {
            "hog_graph_index": hog_index,
            "vertices": vertices,
            "edges": edges,
            "complexity_proxy_n_pow_2": vertices**2,
            "complexity_proxy_n_pow_3": vertices**3,
            "complexity_proxy_n_pow_4": vertices**4,
            "complexity_proxy_edges_x_n2": edges * vertices**2 if not pd.isna(edges) else math.nan,
            "qaoa_statevector_amplitudes_2_pow_n": 2**vertices,
        }
    )

graphs = pd.DataFrame(graph_rows)

print(f"Graphs parsed: {len(graphs)}")
print("Vertex distribution:")
print(graphs["vertices"].value_counts().sort_index().to_string())
print()


# ============================================================
# CSV COLUMN DETECTION
# ============================================================

def find_col(df: pd.DataFrame, candidates: list[str], override: str | None = None, required: bool = True) -> str | None:
    if override is not None:
        if override not in df.columns:
            raise KeyError(f"Override column not found: {override}\nAvailable columns:\n{list(df.columns)}")
        return override

    lower_to_actual = {c.lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower_to_actual:
            return lower_to_actual[candidate.lower()]

    if required:
        raise KeyError(f"Could not find any of {candidates}\nAvailable columns:\n{list(df.columns)}")

    return None


def choose_time_col(df: pd.DataFrame, override: str | None = None) -> str:
    if override is not None:
        if override not in df.columns:
            raise KeyError(f"TIME_COL_OVERRIDE not found: {override}\nAvailable columns:\n{list(df.columns)}")
        return override

    preferred = [
        "qaoa_runtime_seconds",
        "qaoa_time_seconds",
        "qaoa_elapsed_seconds",
        "optimizer_runtime_seconds",
        "optimiser_runtime_seconds",
        "optimization_runtime_seconds",
        "optimisation_runtime_seconds",
        "runtime_seconds",
        "elapsed_seconds",
        "duration_seconds",
        "wall_time_seconds",
        "time_seconds",
    ]

    lower_to_actual = {c.lower(): c for c in df.columns}

    for name in preferred:
        if name.lower() in lower_to_actual:
            col = lower_to_actual[name.lower()]
            values = pd.to_numeric(df[col], errors="coerce").dropna()
            values = values[values > 0]

            if len(values) == 0:
                continue

            # Avoid global elapsed-time columns by rejecting obviously impossible per-instance values.
            valid_fraction = (values <= MAX_VALID_RUNTIME_SECONDS).mean()
            if valid_fraction >= 0.8:
                return col

    candidates = []
    for col in df.columns:
        low = col.lower()
        if "time" not in low and "runtime" not in low and "elapsed" not in low and "duration" not in low:
            continue
        if "start" in low or "timestamp" in low or "global" in low:
            continue

        values = pd.to_numeric(df[col], errors="coerce").dropna()
        values = values[values > 0]

        if len(values) == 0:
            continue

        valid_fraction = (values <= MAX_VALID_RUNTIME_SECONDS).mean()
        median = values.median()
        candidates.append((valid_fraction, -median, col))

    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][2]

    raise KeyError(
        "Could not auto-detect a runtime column. Set TIME_COL_OVERRIDE manually.\n"
        f"Available columns:\n{list(df.columns)}"
    )


df = pd.read_csv(CSV_PATH)

print("CSV columns:")
print(list(df.columns))
print()

time_col = choose_time_col(df, TIME_COL_OVERRIDE)

depth_col = find_col(
    df,
    ["depth", "p", "qaoa_depth"],
    DEPTH_COL_OVERRIDE,
    required=True,
)

repeat_col = find_col(
    df,
    ["benchmark_repeat_idx", "repeat_idx", "repeat", "rep", "repetition"],
    REPEAT_COL_OVERRIDE,
    required=False,
)

explicit_vertex_col = find_col(
    df,
    ["n_vertices", "num_vertices", "number_of_vertices", "numberOf_vertices", "vertices"],
    VERTEX_COL_OVERRIDE,
    required=False,
)

hog_index_col = find_col(
    df,
    ["hog_graph_index", "graph_index", "graph_idx"],
    HOG_INDEX_COL_OVERRIDE,
    required=False,
)

# If there is no explicit vertex column, decide whether column n is vertices or HOG index.
n_col = find_col(df, ["n"], None, required=False)

print("Detected raw columns:")
print(f"  time_col            = {time_col}")
print(f"  depth_col           = {depth_col}")
print(f"  repeat_col          = {repeat_col}")
print(f"  explicit_vertex_col = {explicit_vertex_col}")
print(f"  hog_index_col       = {hog_index_col}")
print(f"  n_col               = {n_col}")
print()


# ============================================================
# NORMALIZE EMPIRICAL ROWS
# ============================================================

work = df.copy()
work["runtime_seconds"] = pd.to_numeric(work[time_col], errors="coerce")
work["depth"] = pd.to_numeric(work[depth_col], errors="coerce")

if repeat_col is not None:
    work["repeat"] = pd.to_numeric(work[repeat_col], errors="coerce")
else:
    work["repeat"] = 1

if explicit_vertex_col is not None:
    work["vertices"] = pd.to_numeric(work[explicit_vertex_col], errors="coerce")
    work["hog_graph_index"] = pd.NA

elif hog_index_col is not None:
    work["hog_graph_index"] = pd.to_numeric(work[hog_index_col], errors="coerce")
    work = work.merge(
        graphs[["hog_graph_index", "vertices"]],
        on="hog_graph_index",
        how="left",
    )

elif n_col is not None:
    n_values = pd.to_numeric(work[n_col], errors="coerce")
    finite_n = n_values.dropna()

    if len(finite_n) == 0:
        raise RuntimeError("Column n exists but contains no numeric values.")

    max_n = int(finite_n.max())
    min_n = int(finite_n.min())

    # If n exceeds the maximum vertex count in the HOG file, it cannot be vertices.
    # Then it is almost certainly the 0-based HOG graph index.
    if max_n > int(graphs["vertices"].max()):
        work["hog_graph_index"] = n_values
        work = work.merge(
            graphs[["hog_graph_index", "vertices"]],
            on="hog_graph_index",
            how="left",
        )
        print("Interpreting CSV column 'n' as HOG graph index.")
    else:
        work["vertices"] = n_values
        work["hog_graph_index"] = pd.NA
        print("Interpreting CSV column 'n' as vertex count.")

else:
    raise RuntimeError(
        "Could not determine vertices. Need one of: explicit vertex column, HOG index column, or n."
    )

work = work.dropna(subset=["vertices", "depth", "runtime_seconds", "repeat"]).copy()
work["vertices"] = work["vertices"].astype(int)
work["depth"] = work["depth"].astype(int)
work["repeat"] = work["repeat"].astype(int)

# Keep only plausible per-instance runtimes.
before_runtime_filter = len(work)
work = work[
    (work["runtime_seconds"] > 0)
    & (work["runtime_seconds"] <= MAX_VALID_RUNTIME_SECONDS)
].copy()
after_runtime_filter = len(work)

if after_runtime_filter == 0:
    raise RuntimeError(
        "No empirical rows left after filtering runtime_seconds. "
        "The selected time column is probably not a per-instance runtime. "
        "Set TIME_COL_OVERRIDE manually."
    )

print(f"Empirical rows before runtime filter: {before_runtime_filter}")
print(f"Empirical rows after runtime filter:  {after_runtime_filter}")
print(f"Dropped rows above {MAX_VALID_RUNTIME_SECONDS}s or <= 0: {before_runtime_filter - after_runtime_filter}")
print()

print("Runtime sanity check:")
print(work["runtime_seconds"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())
print()

print("Largest empirical runtimes used:")
display(
    work.sort_values("runtime_seconds", ascending=False)
    [["vertices", "depth", "repeat", "runtime_seconds"]]
    .head(20)
)


# ============================================================
# RUNTIME MODEL
# ============================================================

def robust_median_seconds(series: pd.Series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    values = values[(values > 0) & (values <= MAX_VALID_RUNTIME_SECONDS)]

    if len(values) == 0:
        return math.nan

    # Use median, not max. This avoids stuck/outlier runs dominating estimates.
    return float(values.median())


runtime_by_vertices_depth = (
    work.groupby(["vertices", "depth"])["runtime_seconds"]
    .apply(robust_median_seconds)
    .to_dict()
)

runtime_by_vertices = (
    work.groupby("vertices")["runtime_seconds"]
    .apply(robust_median_seconds)
    .to_dict()
)

runtime_by_depth = (
    work.groupby("depth")["runtime_seconds"]
    .apply(robust_median_seconds)
    .to_dict()
)

global_runtime = robust_median_seconds(work["runtime_seconds"])


def estimate_runtime(vertices: int, depth: int) -> tuple[float, str]:
    if (vertices, depth) in runtime_by_vertices_depth:
        return runtime_by_vertices_depth[(vertices, depth)], "median_by_vertices_depth"

    if vertices in runtime_by_vertices:
        return runtime_by_vertices[vertices], "median_by_vertices"

    if depth in runtime_by_depth:
        return runtime_by_depth[depth], "median_by_depth"

    return global_runtime, "global_median"


model_preview = (
    work.groupby(["vertices", "depth"])["runtime_seconds"]
    .agg(count="count", median="median", mean="mean", max="max")
    .reset_index()
    .sort_values(["vertices", "depth"])
)

print("Runtime model preview:")
display(model_preview)


# ============================================================
# EXACT COMPLETION KEYS, WHERE AVAILABLE
# ============================================================

completed_keys = set()

if "hog_graph_index" in work.columns and work["hog_graph_index"].notna().any():
    exact = work.dropna(subset=["hog_graph_index"]).copy()
    exact["hog_graph_index"] = exact["hog_graph_index"].astype(int)

    completed_keys = set(
        zip(
            exact["hog_graph_index"].astype(int),
            exact["repeat"].astype(int),
            exact["depth"].astype(int),
        )
    )


# ============================================================
# OUTPUT: ONE ROW PER HOG GRAPH / SEED / DEPTH
# ============================================================

rows = []

for _, graph in graphs.iterrows():
    hog_index = int(graph["hog_graph_index"])

    if hog_index < FIRST_HOG_INDEX:
        continue

    vertices = int(graph["vertices"])
    edges = graph["edges"]

    for repeat in range(1, NUM_REPEATS + 1):
        seed = BASE_SEED + (repeat - 1) * SEED_STEP

        for depth in DEPTHS:
            estimated_seconds, method = estimate_runtime(vertices, depth)
            is_completed_in_source_csv = (hog_index, repeat, depth) in completed_keys

            rows.append(
                {
                    "hog_graph_index": hog_index,
                    "vertices": vertices,
                    "edges": edges,
                    "lasserre_level": LASSERRE_LEVEL,
                    "initial_solver_level_M": INITIAL_SOLVER_LEVEL_M,
                    "repeat": repeat,
                    "seed": seed,
                    "depth": depth,
                    "is_completed_in_source_csv": is_completed_in_source_csv,
                    "complexity_proxy_n_pow_2": graph["complexity_proxy_n_pow_2"],
                    "complexity_proxy_n_pow_3": graph["complexity_proxy_n_pow_3"],
                    "complexity_proxy_n_pow_4": graph["complexity_proxy_n_pow_4"],
                    "complexity_proxy_edges_x_n2": graph["complexity_proxy_edges_x_n2"],
                    "qaoa_statevector_amplitudes_2_pow_n": graph["qaoa_statevector_amplitudes_2_pow_n"],
                    "estimated_qaoa_seconds": estimated_seconds,
                    "estimated_qaoa_minutes": estimated_seconds / 60,
                    "estimate_method": method,
                }
            )

estimate_df = pd.DataFrame(rows)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
estimate_df.to_csv(OUT_PATH, index=False)

summary_df = (
    estimate_df.groupby(["hog_graph_index", "vertices", "edges"], dropna=False)
    .agg(
        rows=("estimated_qaoa_seconds", "size"),
        completed_rows_in_source_csv=("is_completed_in_source_csv", "sum"),
        estimated_qaoa_seconds_total=("estimated_qaoa_seconds", "sum"),
        estimated_qaoa_minutes_total=("estimated_qaoa_minutes", "sum"),
        estimated_qaoa_seconds_mean=("estimated_qaoa_seconds", "mean"),
        estimated_qaoa_seconds_max=("estimated_qaoa_seconds", "max"),
    )
    .reset_index()
)

summary_df.to_csv(SUMMARY_PATH, index=False)


# ============================================================
# FINAL SUMMARY
# ============================================================

serial_hours_all = estimate_df["estimated_qaoa_seconds"].sum() / 3600

remaining_df = estimate_df[~estimate_df["is_completed_in_source_csv"]].copy()
serial_hours_remaining = remaining_df["estimated_qaoa_seconds"].sum() / 3600

print("Wrote:")
print(OUT_PATH.resolve())
print(SUMMARY_PATH.resolve())
print()

print(f"Rows written: {len(estimate_df)}")
print(f"Graphs covered: {estimate_df['hog_graph_index'].nunique()}")
print(f"Completed rows detected from source CSV: {estimate_df['is_completed_in_source_csv'].sum()}")
print()

print(f"Serial estimated QAOA total, all rows:       {serial_hours_all:.2f} h")
print(f"Wall estimated QAOA total with {WORKERS} workers: {serial_hours_all / WORKERS:.2f} h")
print()

print(f"Serial estimated QAOA remaining:             {serial_hours_remaining:.2f} h")
print(f"Wall estimated QAOA remaining with {WORKERS} workers: {serial_hours_remaining / WORKERS:.2f} h")
print()

print("Estimate method counts:")
print(estimate_df["estimate_method"].value_counts().to_string())
print()

print("Largest estimated rows:")
display(
    estimate_df.sort_values("estimated_qaoa_seconds", ascending=False)
    .head(20)
)

Using CSV:
/Users/julian/University/MasterThesis/QAOA/Results/logs/adam/qaoa_results_adam_20260617_173540_pid708337.csv

Using adjacency-list file:
/Users/julian/University/MasterThesis/QAOA/Code/HOG_graphs/triangleFree_connected_4to8vertices_176instances_AdjacencyList.txt

Graphs parsed: 176
Vertex distribution:
vertices
5      3
6      6
7     19
8     44
9    104

CSV columns:
['run_id', 'n', 'm', 'p', 'precision/iterations', 'singlet_injection', 'warm_start', 'parameter_vector', 'optimal_result', 'sdp_objective_value_step1', 'sdp_objective_value_king_normalized_step1', 'algorithm17_actual_energy', 'algorithm17_lower_bound_energy', 'initial_ws_energy_prodStates_step2', 'initial_sdp_statevector_energy', 'initial_sdp_statevec_ratio', 'initial_ws_energy_010101', 'QAOA_improvement_over_SDP_statevectorEnergy', 'QAOA_improvement_over_SDP_prodStatesEnergy', 'result', 'result_010101', 'approx_ratio', 'approx_ratio_010101', 'diff. approx. ratio', 'sdp ws greater', 'duration_seconds', 'durati

,vertices,depth,repeat,runtime_seconds
588,9,1,8,2513.585085
586,9,1,7,2503.178182
570,9,1,1,2487.018152
587,9,1,9,2485.782274
589,9,1,10,2482.377949
585,9,1,6,2478.550370
573,9,1,4,2465.675719
571,9,1,2,2462.812486
574,9,1,5,2462.116577
572,9,1,3,2458.714654


Runtime model preview:


,vertices,depth,count,median,mean,max
0,5,1,20,66.558580,59.837004,75.215610
1,5,3,20,107.501953,114.169791,168.038488
2,5,5,20,263.959711,274.423695,399.580449
3,6,1,20,104.157947,101.055639,116.130004
4,6,3,20,112.457314,129.820416,199.005845
5,6,5,20,263.520020,324.728599,466.132566
6,7,1,75,224.618665,219.141929,289.596076
7,7,3,70,170.762472,205.551791,474.978134
8,7,5,70,393.958378,448.665460,727.357069
9,8,1,90,656.835817,664.540307,822.981548


Wrote:
/Users/julian/University/MasterThesis/QAOA/Results/logs/adam/L2M2_runtime_estimate_by_instance_seed_depth.csv
/Users/julian/University/MasterThesis/QAOA/Results/logs/adam/L2M2_runtime_estimate_by_graph_summary.csv

Rows written: 3750
Graphs covered: 125
Completed rows detected from source CSV: 0

Serial estimated QAOA total, all rows:       825.44 h
Wall estimated QAOA total with 5 workers: 165.09 h

Serial estimated QAOA remaining:             825.44 h
Wall estimated QAOA remaining with 5 workers: 165.09 h

Estimate method counts:
estimate_method
median_by_vertices_depth    3750

Largest estimated rows:


,hog_graph_index,vertices,edges,lasserre_level,initial_solver_level_M,repeat,seed,depth,is_completed_in_source_csv,complexity_proxy_n_pow_2,complexity_proxy_n_pow_3,complexity_proxy_n_pow_4,complexity_proxy_edges_x_n2,qaoa_statevector_amplitudes_2_pow_n,estimated_qaoa_seconds,estimated_qaoa_minutes,estimate_method
1875,113,9,NaN,2,2,6,5000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2628,138,9,NaN,2,2,7,6000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2709,141,9,NaN,2,2,4,3000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
1905,114,9,NaN,2,2,6,5000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2706,141,9,NaN,2,2,3,2000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
1170,90,9,NaN,2,2,1,42,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2703,141,9,NaN,2,2,2,1000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2700,141,9,NaN,2,2,1,42,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
1908,114,9,NaN,2,2,7,6000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
2637,138,9,NaN,2,2,10,9000042,1,False,81.0,729.0,6561.0,NaN,512.0,1776.253803,29.60423,median_by_vertices_depth
